# IBRD World Loans Risk Analysis

This notebook identifies high-risk loans, explains key risk drivers, and highlights anomalous repayment patterns in the IBRD portfolio.



In [18]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from scipy import stats
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Resolve the project root robustly
candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
project_root = next(
    (root for root in candidate_roots if (root / "data" / "processed" / "ibrd_clean.csv").exists()),
    Path("/home/rigii/ATA").resolve(),
)

DATA_PATH = project_root / "data" / "processed" / "ibrd_clean.csv"
FIGURES_DIR = project_root / "reports" / "figures"
FEATURES_DIR = project_root / "data" / "features"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.write_image(str(path), width=1400, height=900, scale=2)
    print(f"Saved chart: {path}")
    return path

print(f"Project root: {project_root}")
print(f"Data path: {DATA_PATH}")
print(f"Figures dir: {FIGURES_DIR}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")

df["risk_score_original"] = pd.to_numeric(df.get("risk_score", 0), errors="coerce").fillna(0)
if df["risk_score_original"].max() <= 1.5:
    df["risk_score"] = (df["risk_score_original"] * 6).clip(0, 6)
else:
    df["risk_score"] = df["risk_score_original"].clip(0, 6)

for col in [
    "Original Principal Amount (US$)",
    "Cancelled Amount (US$)",
    "Undisbursed Amount (US$)",
    "Disbursed Amount (US$)",
    "Repaid to IBRD (US$)",
    "Due to IBRD (US$)",
    "Interest Rate",
    "loan_age_years",
    "disbursed_percent",
    "cancelled_percent",
    "undisbursed_percent",
    "repayment_ratio",
    "outstanding_ratio",
    "net_disbursed",
    "approval_year",
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "loan_size_category" not in df.columns:
    df["loan_size_category"] = pd.cut(
        df["Original Principal Amount (US$)"].fillna(0),
        bins=[0, 1_000_000, 10_000_000, 100_000_000, 500_000_000, np.inf],
        labels=["Very Small", "Small", "Medium", "Large", "Very Large"],
        right=False,
    )

if "age_category" not in df.columns:
    df["age_category"] = pd.cut(
        df["loan_age_years"].fillna(0),
        bins=[0, 5, 10, 15, 20, np.inf],
        labels=["0-5 yrs", "5-10 yrs", "10-15 yrs", "15-20 yrs", "20+ yrs"],
        right=False,
    )

print(df[["Loan Number", "Region", "Country / Economy", "risk_score", "repayment_ratio", "loan_age_years"]].head().to_string(index=False))
print(f"Risk score range: {df['risk_score'].min():.2f} to {df['risk_score'].max():.2f}")



Project root: /home/rigii/ATA
Data path: /home/rigii/ATA/data/processed/ibrd_clean.csv
Figures dir: /home/rigii/ATA/reports/figures
Loaded dataset with 9518 rows and 54 columns
Loan Number                Region Country / Economy  risk_score  repayment_ratio  loan_age_years
  IBRD87030 EAST ASIA AND PACIFIC             China         3.9         0.171472        9.522245
  IBRD87040 EAST ASIA AND PACIFIC             China         3.9         0.045900        9.459274
  IBRD87200 EAST ASIA AND PACIFIC             China         3.9         0.138326        9.418207
  IBRD87440 EAST ASIA AND PACIFIC             China         3.9         0.137678        9.311431
  IBRD87660 EAST ASIA AND PACIFIC             China         3.9         0.071800        9.169062
Risk score range: 0.00 to 6.00


In [19]:
risk_counts = df["risk_score"].round().astype(int).clip(0, 6)
risk_counts = risk_counts.value_counts().reindex(range(0, 7), fill_value=0).sort_index()

fig = px.bar(
    x=list(risk_counts.index),
    y=risk_counts.values,
    title="Risk Score Distribution (0–6)",
    labels={"x": "Risk Score", "y": "Number of Loans"},
    color_discrete_sequence=["#4c78a8"],
)
fig.update_layout(template="plotly_white", xaxis=dict(dtick=1))
fig.show()
save_figure(fig, "loan_risk_score_distribution.png")
print(risk_counts.to_string())



Saved chart: /home/rigii/ATA/reports/figures/loan_risk_score_distribution.png
risk_score
0    6623
1     163
2     214
3     730
4    1239
5     329
6     220


In [20]:
high_risk = df[df["risk_score"] >= 3].copy()
print(f"High-risk loans: {len(high_risk)}")

region_summary = (
    high_risk.groupby("Region", dropna=False)
    .agg(loan_count=("Loan Number", "count"), total_commitments=("Original Principal Amount (US$)", "sum"))
    .sort_values(["loan_count", "total_commitments"], ascending=[False, False])
    .head(10)
    .reset_index()
)

country_summary = (
    high_risk.groupby("Country / Economy", dropna=False)
    .agg(loan_count=("Loan Number", "count"), total_commitments=("Original Principal Amount (US$)", "sum"))
    .sort_values(["loan_count", "total_commitments"], ascending=[False, False])
    .head(10)
    .reset_index()
)

print("Top high-risk regions:")
print(region_summary.to_string(index=False))
print("Top high-risk countries:")
print(country_summary.to_string(index=False))

fig = px.bar(
    region_summary,
    x="Region",
    y="loan_count",
    title="High-Risk Loans by Region",
    labels={"loan_count": "High-Risk Loan Count", "Region": "Region"},
    color="loan_count",
    color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_white")
fig.show()
save_figure(fig, "high_risk_loans_by_region.png")

high_risk.to_csv(FEATURES_DIR / "high_risk_loans.csv", index=False)
print(f"Saved high-risk loans table: {FEATURES_DIR / 'high_risk_loans.csv'}")



High-risk loans: 2402
Top high-risk regions:
                       Region  loan_count  total_commitments
  LATIN AMERICA AND CARIBBEAN         697       1.344516e+11
      EUROPE AND CENTRAL ASIA         584       9.993881e+10
        EAST ASIA AND PACIFIC         441       9.146283e+10
MID EAST,NORTH AFRICA,AFG,PAK         324       7.138972e+10
                   SOUTH ASIA         192       4.942140e+10
  EASTERN AND SOUTHERN AFRICA         111       2.609543e+10
   WESTERN AND CENTRAL AFRICA          53       7.994700e+09
Top high-risk countries:
Country / Economy  loan_count  total_commitments
            India         179       4.783250e+10
            China         173       2.287069e+10
        Indonesia         148       3.770572e+10
           Brazil         142       2.969789e+10
        Argentina         106       2.399466e+10
          Turkiye         101       3.159267e+10
          Morocco          91       1.948568e+10
      Philippines          70       2.526674e+10
 

Saved chart: /home/rigii/ATA/reports/figures/high_risk_loans_by_region.png
Saved high-risk loans table: /home/rigii/ATA/data/features/high_risk_loans.csv


In [21]:
age_risk = (
    df.groupby("age_category", dropna=False)["risk_score"]
    .mean()
    .reset_index(name="avg_risk_score")
    .sort_values("avg_risk_score", ascending=False)
)

fig = px.line(
    age_risk,
    x="age_category",
    y="avg_risk_score",
    markers=True,
    title="Risk by Loan Age",
    labels={"age_category": "Loan Age Category", "avg_risk_score": "Average Risk Score"},
)
fig.update_traces(line=dict(width=3))
fig.update_layout(template="plotly_white")
fig.show()
fig.write_image(str(FIGURES_DIR / "risk_by_loan_age.png"), width=1400, height=900, scale=2)
print("Saved chart: ", FIGURES_DIR / "risk_by_loan_age.png")



Saved chart:  /home/rigii/ATA/reports/figures/risk_by_loan_age.png


In [22]:
size_box = df.dropna(subset=["loan_size_category"]).copy()
fig = px.box(
    size_box,
    x="loan_size_category",
    y="risk_score",
    title="Risk Score by Loan Size Category",
    labels={"loan_size_category": "Loan Size Category", "risk_score": "Risk Score"},
    color="loan_size_category",
)
fig.update_layout(template="plotly_white")
fig.show()
save_figure(fig, "risk_by_loan_size.png")



Saved chart: /home/rigii/ATA/reports/figures/risk_by_loan_size.png


PosixPath('/home/rigii/ATA/reports/figures/risk_by_loan_size.png')

In [23]:
feature_cols = [
    "Original Principal Amount (US$)",
    "Disbursed Amount (US$)",
    "repayment_ratio",
    "cancelled_percent",
    "loan_age_years",
]

X = df[feature_cols].fillna(0)
clf = IsolationForest(contamination=0.05, random_state=42)
df["anomaly"] = clf.fit_predict(X)

print(f"Anomaly count: {(df['anomaly'] == -1).sum()}")
print(f"Normal count: {(df['anomaly'] == 1).sum()}")

loan_anomalies = df[df["anomaly"] == -1].copy()
loan_anomalies = loan_anomalies.sort_values(["risk_score", "Original Principal Amount (US$)"], ascending=[False, False]).head(20)
print("Top 20 anomalous loans:")
print(loan_anomalies[[
    "Loan Number",
    "Country / Economy",
    "Region",
    "Original Principal Amount (US$)",
    "Disbursed Amount (US$)",
    "repayment_ratio",
    "cancelled_percent",
    "loan_age_years",
    "risk_score",
    "anomaly",
]].to_string(index=False))

scatter_df = df[[
    "Loan Number",
    "Country / Economy",
    "Region",
    "Disbursed Amount (US$)",
    "Repaid to IBRD (US$)",
    "anomaly",
]].dropna(subset=["Disbursed Amount (US$)", "Repaid to IBRD (US$)"]).copy()
scatter_df["anomaly_label"] = scatter_df["anomaly"].map({-1: "Anomaly", 1: "Normal"})

fig = px.scatter(
    scatter_df,
    x="Disbursed Amount (US$)",
    y="Repaid to IBRD (US$)",
    color="anomaly_label",
    color_discrete_map={"Anomaly": "#d62728", "Normal": "#1f77b4"},
    title="Isolation Forest Loan Anomalies",
    opacity=0.7,
    labels={"x": "Disbursed Amount (US$)", "y": "Repaid to IBRD (US$)", "anomaly_label": "Loan Status"},
)
fig.update_layout(template="plotly_white")
fig.show()
save_figure(fig, "anomaly_scatter_plot.png")



Anomaly count: 476
Normal count: 9042
Top 20 anomalous loans:
Loan Number  Country / Economy                      Region  Original Principal Amount (US$)  Disbursed Amount (US$)  repayment_ratio  cancelled_percent  loan_age_years  risk_score  anomaly
  IBRDG2440            Ukraine     EUROPE AND CENTRAL ASIA                     500000000.00                     0.0              0.0              100.0        9.867214         6.0       -1
  IBRD91480         Kazakhstan     EUROPE AND CENTRAL ASIA                     500000000.00                     0.0              0.0              100.0        6.162902         6.0       -1
  IBRDB0200             Mexico LATIN AMERICA AND CARIBBEAN                     500000000.00                     0.0              0.0              100.0       39.457906         6.0       -1
  IBRD93270               Peru LATIN AMERICA AND CARIBBEAN                     500000000.00                     0.0              0.0              100.0        4.695414         6.0   

Saved chart: /home/rigii/ATA/reports/figures/anomaly_scatter_plot.png


PosixPath('/home/rigii/ATA/reports/figures/anomaly_scatter_plot.png')

In [24]:
country_repayment = (
    df.groupby("Country / Economy", dropna=False)["repayment_ratio"]
    .mean()
    .reset_index(name="mean_repayment_ratio")
)

if not country_repayment.empty:
    mean_repayment = country_repayment["mean_repayment_ratio"].mean()
    std_repayment = country_repayment["mean_repayment_ratio"].std(ddof=0)
    if pd.notna(std_repayment) and std_repayment > 0:
        country_repayment["z_score"] = (country_repayment["mean_repayment_ratio"] - mean_repayment) / std_repayment
    else:
        country_repayment["z_score"] = 0
    worst_countries = country_repayment[country_repayment["z_score"] < -2].sort_values("z_score").head(10)
    print("Worst performing countries by repayment z-score:")
    print(worst_countries.to_string(index=False))
else:
    worst_countries = pd.DataFrame(columns=["Country / Economy", "mean_repayment_ratio", "z_score"])
    print("No countries available for z-score analysis.")



Worst performing countries by repayment z-score:
Country / Economy  mean_repayment_ratio   z_score
           Bhutan              0.000000 -2.456214
       Mozambique              0.000000 -2.456214
           France              0.000152 -2.455665
            Malta              0.005642 -2.435845
          Belgium              0.018087 -2.390918
         Mongolia              0.026115 -2.361939
           Angola              0.041038 -2.308065
         Suriname              0.053714 -2.262305
          Georgia              0.102961 -2.084523
      Netherlands              0.107150 -2.069400


In [25]:
cancelled = df[df.get("is_cancelled", False) == True].copy()
print(f"Cancelled loans: {len(cancelled)}")

if not cancelled.empty:
    fig = px.histogram(
        cancelled,
        x="cancelled_percent",
        nbins=25,
        title="Cancelled Loan Share Distribution",
        labels={"cancelled_percent": "Cancelled % of Original Commitment"},
        color_discrete_sequence=["#ff7f0e"],
    )
    fig.update_layout(template="plotly_white")
    fig.show()
    save_figure(fig, "cancellation_analysis.png")
    
    country_cancel = (
        cancelled.groupby("Country / Economy", dropna=False)
        .agg(cancelled_loans=("Loan Number", "count"), avg_cancelled_percent=("cancelled_percent", "mean"))
        .sort_values(["avg_cancelled_percent", "cancelled_loans"], ascending=[False, False])
        .head(10)
        .reset_index()
    )
    print("Top countries by cancellation rate:")
    print(country_cancel.to_string(index=False))
else:
    print("No cancelled loans in the dataset.")



Cancelled loans: 220


Saved chart: /home/rigii/ATA/reports/figures/cancellation_analysis.png
Top countries by cancellation rate:
      Country / Economy  cancelled_loans  avg_cancelled_percent
     Yugoslavia, Former               18                  100.0
                  China               14                  100.0
                   Peru                9                  100.0
               Pakistan                7                  100.0
Egypt, Arab Republic Of                6                  100.0
                Lebanon                5                  100.0
                 Poland                5                  100.0
                Ukraine                5                  100.0
               Colombia                4                  100.0
                Ecuador                4                  100.0


In [28]:
rate_df = df[["Interest Rate", "risk_score"]].copy()
rate_df["Interest Rate"] = pd.to_numeric(rate_df["Interest Rate"], errors="coerce")
rate_df = rate_df.dropna(subset=["Interest Rate"]).copy()
rate_df = rate_df[rate_df["Interest Rate"].between(0, 25)]

if not rate_df.empty:
    # Avoid Plotly's `trendline="ols"` because it requires statsmodels.
    slope, intercept, _, _, _ = stats.linregress(rate_df["Interest Rate"], rate_df["risk_score"])
    x_line = np.linspace(rate_df["Interest Rate"].min(), rate_df["Interest Rate"].max(), 200)
    y_line = intercept + slope * x_line

    fig = px.scatter(
        rate_df,
        x="Interest Rate",
        y="risk_score",
        title="Interest Rate vs Risk Score",
        labels={"Interest Rate": "Interest Rate (%)", "risk_score": "Risk Score"},
        opacity=0.7,
    )
    trend = px.line(x=x_line, y=y_line)
    trend.data[0].update(name="Linear trend", line=dict(color="red", width=2), mode="lines")
    fig.add_trace(trend.data[0])
    fig.update_layout(template="plotly_white")
    fig.show()
    save_figure(fig, "interest_rate_vs_risk.png")
else:
    print("No usable interest rate values to plot.")



Saved chart: /home/rigii/ATA/reports/figures/interest_rate_vs_risk.png


In [29]:
region_summary = (
    high_risk.groupby("Region", dropna=False)
    .agg(loan_count=("Loan Number", "count"), avg_risk=("risk_score", "mean"), total_commitment=("Original Principal Amount (US$)", "sum"))
    .sort_values(["loan_count", "avg_risk"], ascending=[False, False])
    .reset_index()
)

size_summary = df.groupby("loan_size_category", dropna=False)["risk_score"].mean().reset_index(name="avg_risk")

region_top = region_summary.head(1).to_dict("records")[0] if not region_summary.empty else {"Region": "N/A", "loan_count": 0, "avg_risk": 0.0}
country_top = worst_countries.head(1).to_dict("records")[0] if not worst_countries.empty else {"Country / Economy": "N/A", "z_score": 0.0}
size_top = size_summary.sort_values("avg_risk", ascending=False).head(1).to_dict("records")[0] if not size_summary.empty else {"loan_size_category": "N/A", "avg_risk": 0.0}

summary_md = f"""
## Summary of findings

- The highest-risk region is **{region_top['Region']}** with **{region_top['loan_count']}** high-risk loans and an average risk score of **{region_top['avg_risk']:.2f}**.
- The most compromised repayment profile appears in **{country_top['Country / Economy']}** with a repayment z-score of **{country_top['z_score']:.2f}**.
- Larger loans generally appear to be riskier, with **{size_top['loan_size_category']}** showing the highest average risk score at **{size_top['avg_risk']:.2f}**.
- Recommendations: prioritize watchlist review for the highest-risk regions, investigate anomalous country-level repayment patterns, and apply tighter monitoring to large, highly cancelled or low-repayment loans.
"""
display(Markdown(summary_md))




## Summary of findings

- The highest-risk region is **LATIN AMERICA AND CARIBBEAN** with **697** high-risk loans and an average risk score of **3.85**.
- The most compromised repayment profile appears in **Bhutan** with a repayment z-score of **-2.46**.
- Larger loans generally appear to be riskier, with **Mega** showing the highest average risk score at **2.88**.
- Recommendations: prioritize watchlist review for the highest-risk regions, investigate anomalous country-level repayment patterns, and apply tighter monitoring to large, highly cancelled or low-repayment loans.


In [ ]:
print("Notebook execution completed successfully.")



Notebook execution completed successfully.
